# Tutorial: Tools, Chains, Agents — Core Concepts (CRM Mini-App)

In this tutorial, we build a small **in-memory CRM application** to understand three LangChain concepts:

- **Tool** — a function that performs an action, such as adding or retrieving a contact.
- **Chain** — a predictable sequence of LLM processing steps.
- **Agent / Router** — an LLM-assisted component that determines which action should be performed.

The application uses **OpenRouter** as the model provider and current LangChain APIs.

> **Important:** This notebook intentionally does not use the legacy `LLMChain` API. Chains are built with **LangChain Expression Language (LCEL)** using the `|` operator and executed with `.invoke()`.

## Learning objectives

By the end of this tutorial, you will be able to:

1. Explain the difference between a tool, chain, and agent.
2. Create Python functions that behave as application tools.
3. Build an LCEL chain with `PromptTemplate`, `ChatOpenAI`, and `StrOutputParser`.
4. Use `.invoke()` to execute a chain.
5. Use an LLM to classify user intent.
6. Route a request to the appropriate CRM function.
7. Understand the difference between a simple LLM-based router and a full tool-calling agent.

## Architecture

```text
User Request
     |
     v
Intent Classification Chain
     |
     +---- add ----> add_contact() ----+
     |                                 |
     +---- read ---> get_contact() ----+--> Formatting Chain --> Response
     |
     +---- unknown --> Help Message
```

The CRM data is stored only in memory. It is not written to a database.


## Step 1: Install the Required Packages

The original notebook installed several packages that are not needed for this lab.

For this tutorial we only need:

- `langchain`
- `langchain-openai`

We do **not** need `langchain-community`, `langchain-classic`, `duckduckgo-search`, `wikipedia`, `numexpr`, or `python-dotenv` for the examples in this notebook.


In [ ]:
%pip install -qU langchain langchain-openai


## Step 2: Import Libraries and Configure OpenRouter

OpenRouter exposes an OpenAI-compatible API. LangChain's `ChatOpenAI` integration can therefore be configured with OpenRouter's API endpoint.

The API key is requested using `getpass()`, so it is not displayed.

**Security reminder:** Never hard-code an API key into a notebook that will be shared or committed to GitHub.


In [ ]:
from getpass import getpass
from typing import Dict, Optional, Literal, Tuple
import json
import re

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key (hidden): ")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
)

output_parser = StrOutputParser()

print("OpenRouter model configured successfully.")


## Step 3: Understand the Three Concepts

### Tool

A **tool** is a callable function that performs an action.

Examples:

```text
add_contact()
get_contact()
search_customer()
send_email()
```

### Chain

A **chain** connects known processing steps.

For example:

```text
Prompt → Model → Output Parser
```

### Agent

An **agent** can determine what action should be taken based on the user's request and available tools.

In this tutorial, we first implement a simple **LLM-based router**. This makes the concept easy to understand before moving to a full LangChain tool-calling agent.

> The router in this notebook is intentionally simple. It demonstrates agent-like decision making but is not the same as LangChain's full tool-calling agent runtime.


## Step 4: Create the CRM Tools

We create two simple tools over an in-memory dictionary:

- `add_contact(name, email, phone)` — adds or updates a contact.
- `get_contact(name)` — retrieves a contact.


In [ ]:
crm_store: Dict[str, Dict[str, str]] = {}


def add_contact(
    name: str,
    email: Optional[str] = None,
    phone: Optional[str] = None
) -> Dict:
    """Add or update a contact in the in-memory CRM store."""
    if not name or not isinstance(name, str):
        return {"status": "error", "message": "Name is required and must be a string."}

    name = name.strip()
    record = crm_store.get(name, {"name": name})

    if email:
        record["email"] = email.strip()
    if phone:
        record["phone"] = phone.strip()

    crm_store[name] = record

    return {"status": "ok", "message": "Contact saved.", "contact": record}


def get_contact(name: str) -> Dict:
    """Retrieve a contact by name."""
    if not name or not isinstance(name, str):
        return {"status": "error", "message": "Name is required and must be a string."}

    name = name.strip()
    record = crm_store.get(name)

    if record:
        return {"status": "ok", "contact": record}

    return {"status": "not_found", "message": f"No contact found for '{name}'"}


print(add_contact("Alice", email="alice@example.com"))
print(get_contact("Alice"))


## Step 5: Build a Formatting Chain

The CRM tools return Python dictionaries. We use an LLM chain to turn the raw result into a concise user-facing response.

### Modern LCEL pattern

```python
chain = prompt | model | output_parser
```

This replaces the legacy `LLMChain` and `.run()` pattern.


In [ ]:
format_prompt = PromptTemplate.from_template(
    """You are a helpful CRM assistant.

Given the CRM tool result below, produce a concise user-facing response.

Rules:
- If status is "ok" and a contact exists, state the contact name and available email/phone.
- If status is "not_found" or "error", return the message clearly.
- Do not invent missing information.

CRM result:
{tool_result}
"""
)

format_chain = format_prompt | llm | output_parser


def format_tool_result(tool_result: dict) -> str:
    return format_chain.invoke({"tool_result": str(tool_result)})


print(format_tool_result(get_contact("Alice")))


## Step 6: Build an Intent Classification Chain

The application needs to determine whether the user wants to:

- **add** a contact
- **read** a contact
- perform an **unknown** operation

We use a small LLM chain for this classification.


In [ ]:
Intent = Literal["add", "read", "unknown"]

intent_prompt = PromptTemplate.from_template(
    """Determine the user's CRM intent.

Return exactly one word:
- add
- read
- unknown

Use "add" for requests to add, save, or update a contact.
Use "read" for requests to find, retrieve, look up, or show a contact.

User request:
{text}
"""
)

intent_chain = intent_prompt | llm | output_parser


def classify_intent(text: str) -> Intent:
    result = intent_chain.invoke({"text": text}).strip().lower()

    if result.startswith("add"):
        return "add"
    if result.startswith(("read", "lookup", "get", "find", "show")):
        return "read"
    return "unknown"


for example in [
    "Add John Doe with email john@example.com",
    "Find Alice",
    "Delete Bob"
]:
    print(example, "->", classify_intent(example))


## Step 7: Extract Information for an Add Request

Once the intent is `add`, we need to extract:

- name
- email
- phone

The LLM returns JSON, which we then parse in Python.


In [ ]:
extract_add_prompt = PromptTemplate.from_template(
    """Extract contact information from the user's request.

Return ONLY valid JSON using exactly these keys:
{
  "name": string or null,
  "email": string or null,
  "phone": string or null
}

Do not add markdown or explanation.

User request:
{text}
"""
)

extract_add_chain = extract_add_prompt | llm | output_parser


def parse_add_fields(text: str) -> Tuple[Optional[str], Dict[str, Optional[str]]]:
    raw = extract_add_chain.invoke({"text": text}).strip()

    try:
        match = re.search(r"\{.*\}", raw, re.S)
        if not match:
            raise ValueError("No JSON object found.")

        data = json.loads(match.group(0))

        return data.get("name"), {
            "email": data.get("email"),
            "phone": data.get("phone")
        }

    except Exception:
        return None, {"email": None, "phone": None}


print(parse_add_fields(
    "Add John Doe with email john@example.com and phone +1 202 555 0142"
))


## Step 8: Extract a Contact Name for a Read Request

In [ ]:
extract_read_prompt = PromptTemplate.from_template(
    """Extract the contact name that the user wants to look up.

Return ONLY the person's name.
Do not include quotes, explanation, or punctuation.

User request:
{text}
"""
)

extract_read_chain = extract_read_prompt | llm | output_parser


def parse_read_name(text: str) -> str:
    name = extract_read_chain.invoke({"text": text}).strip()
    return re.sub(r'^[\"\']|[\"\']$', "", name).strip()


print(parse_read_name("Find Alice Cooper"))


## Step 9: Build the CRM Router

The router will:

1. Classify the user's intent.
2. If `add`, extract contact fields.
3. Call `add_contact()`.
4. If `read`, extract the contact name.
5. Call `get_contact()`.
6. Pass the result to the formatting chain.
7. Return the final response.

This demonstrates how tools and chains can be composed into a simple agent-like workflow.


In [ ]:
def crm_agent(user_input: str) -> str:
    """Route a natural-language CRM request to the appropriate operation."""

    intent = classify_intent(user_input)

    if intent == "add":
        name, fields = parse_add_fields(user_input)

        if not name:
            return "Please provide a contact name to add or update."

        result = add_contact(
            name=name,
            email=fields.get("email"),
            phone=fields.get("phone")
        )

        return format_tool_result(result)

    if intent == "read":
        name = parse_read_name(user_input)

        if not name:
            return "Please provide the contact name to look up."

        result = get_contact(name)
        return format_tool_result(result)

    return (
        "I can add or look up contacts. "
        "Try: 'Add John Doe with john@example.com' or 'Find John Doe'."
    )


## Step 10: Run the CRM Demo

In [ ]:
print("1. ADD REQUEST")
print(crm_agent(
    "Add Alice Cooper with email alice@co.com and phone +1 202 555 0142"
))

print("\n2. READ REQUEST")
print(crm_agent("Find Alice Cooper"))

print("\n3. UNKNOWN REQUEST")
print(crm_agent("Delete Alice Cooper"))


## Step 11: Inspect the CRM Store

The CRM is deliberately stored in memory so that we can see what the tools actually changed.

No database or external storage is involved.


In [ ]:
crm_store


## Step 12: Test More Requests

Try requests such as:

- `Add Bob Smith with email bob@example.com`
- `Find Bob Smith`
- `Add Sarah Jones with phone +1 555 123 4567`
- `Find Sarah Jones`
- `Find Michael Brown`


In [ ]:
test_requests = [
    "Add Bob Smith with email bob@example.com",
    "Find Bob Smith",
    "Add Sarah Jones with phone +1 555 123 4567",
    "Find Sarah Jones",
    "Find Michael Brown"
]

for request in test_requests:
    print(f"\nUSER: {request}")
    print(f"ASSISTANT: {crm_agent(request)}")


## Step 13: Tool vs Chain vs Agent

| Concept | Role in this lab | Example |
|---|---|---|
| Tool | Performs an application action | `add_contact()` |
| Chain | Performs a known sequence | `prompt → llm → parser` |
| Router / Agent-like logic | Chooses an operation | `add` vs `read` |

### Chain

Use a chain when the sequence is known.

```text
Question → Prompt → LLM → Parser → Answer
```

### Agent

An agent becomes useful when the system needs to decide dynamically which tool or sequence of tools to use.

```text
User
 ↓
Agent
 ├── search customer
 ├── retrieve order
 ├── calculate balance
 └── send email
```

### Important distinction

This tutorial's `crm_agent()` is a **simple LLM-based router**. It demonstrates dynamic routing, but it is not yet a full LangChain tool-calling agent.


## Step 14: Why LCEL?

LangChain Expression Language makes composition explicit.

```python
chain = prompt | llm | output_parser
```

Think of it as:

```text
input
  ↓
prompt
  ↓
model
  ↓
parser
  ↓
output
```

### Legacy pattern

```python
from langchain.chains import LLMChain

chain = LLMChain(llm=llm, prompt=prompt)
result = chain.run({"topic": "AI"})
```

### Modern LCEL pattern

```python
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()
result = chain.invoke({"topic": "AI"})
```

For this lab, use the modern LCEL pattern.


## Step 15: Student Exercise — Add Delete Support

Modify the CRM application to support a third operation: **delete a contact**.

### Requirements

1. Create `delete_contact(name)`.
2. Update intent classification to return `add`, `read`, `delete`, or `unknown`.
3. Route delete requests from `crm_agent()`.
4. Test:

```text
Add David Brown with email david@example.com
Find David Brown
Delete David Brown
Find David Brown
```

### Expected behavior

After deletion, the final lookup should indicate that the contact cannot be found.

### Optional challenge

Add a `list_contacts` operation that returns all contacts currently stored.


In [ ]:
# STUDENT EXERCISE

def delete_contact(name: str) -> Dict:
    # TODO: implement
    pass

# Then update the intent classifier and crm_agent() to support "delete".


## Step 16: Troubleshooting

### `ModuleNotFoundError: No module named 'langchain.prompts'`

Use:

```python
from langchain_core.prompts import PromptTemplate
```

not:

```python
from langchain.prompts import PromptTemplate
```

### `ModuleNotFoundError: No module named 'langchain.chains'`

This notebook does not require `LLMChain`.

Do not import:

```python
from langchain.chains import LLMChain
```

Use:

```python
chain = prompt | llm | output_parser
```

### `.run()` does not work

Use:

```python
chain.invoke(...)
```

instead of the legacy:

```python
chain.run(...)
```

### OpenRouter authentication error

Check that your API key is valid and that:

```text
https://openrouter.ai/api/v1
```

is being used as the `base_url`.

### Model unavailable

The example uses:

```text
openai/gpt-4o-mini
```

If that model is unavailable to your OpenRouter account, replace it with a model currently available to you.

### Unexpected model output

LLMs are probabilistic. Production applications should validate model-generated data before using it.


## Summary

In this tutorial, you built a small CRM application using current LangChain patterns and OpenRouter.

You learned:

- **Tools** perform application actions.
- **Chains** connect predictable processing steps.
- **LCEL** uses the `|` operator to compose chains.
- `.invoke()` executes modern LangChain runnables.
- An LLM can classify intent and extract information.
- A simple router can connect natural-language requests to application tools.
- A full LangChain agent is a more advanced abstraction that can dynamically select and call tools.

### Key LCEL pattern

```python
chain = prompt | llm | output_parser
result = chain.invoke({"topic": "LangChain"})
```

### Conceptual takeaway

```text
TOOLS
  ↓
Actions the application can perform

CHAINS
  ↓
Known sequence of processing steps

AGENTS
  ↓
Dynamic decision-making over available actions
```

This foundation can be extended into full tool-calling agents, retrieval-augmented generation, structured outputs, and multi-step AI workflows.
